# Phase 5 — Complexity-Aware Router

This notebook implements the adaptive routing layer:

**BeerQA Question → DeepSeek Router → Standard RAG / Query Decomposition RAG / Multi-Agent RAG**

Routing policy:
- **EASY → Standard RAG**
- **MEDIUM → Query Decomposition RAG**
- **HARD → Multi-Agent RAG**

For fair Dev20 evaluation, the router sees only the original question. It does **not** see the gold answer, gold context, or previous pipeline scores.

Because all three RAG pipelines have already been evaluated on the same Dev20 records, this notebook reuses their saved result files. The router decides which pipeline should handle each question, then the corresponding saved pipeline result is selected to form the adaptive framework result.

## Step 1 — Imports and Paths

In [ ]:
from pathlib import Path
from collections import Counter
import os, json, re, getpass

from openai import OpenAI

In [ ]:
DEV_SAMPLE_SIZE = 20

OUTPUT_DIR = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\processed_data"
)

BEERQA_DEV_PATH = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\beerqa_dev_v1.0.json"
)

STANDARD_RAG_EVAL_PATH = (
    OUTPUT_DIR / f"standard_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

QD_RAG_EVAL_PATH = (
    OUTPUT_DIR / f"query_decomposition_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

MA_RAG_EVAL_PATH = (
    OUTPUT_DIR / f"multi_agent_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

ROUTER_DEV5_PATH = (
    OUTPUT_DIR / "complexity_router_dev5_decisions.jsonl"
)

ROUTER_DEV20_PATH = (
    OUTPUT_DIR / f"complexity_router_dev{DEV_SAMPLE_SIZE}_decisions.jsonl"
)

ADAPTIVE_DEV20_EVAL_PATH = (
    OUTPUT_DIR / f"adaptive_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

print("BeerQA dev:", BEERQA_DEV_PATH.exists())
print("Standard RAG evaluated:", STANDARD_RAG_EVAL_PATH.exists())
print("Query Decomposition evaluated:", QD_RAG_EVAL_PATH.exists())
print("Multi-Agent evaluated:", MA_RAG_EVAL_PATH.exists())

## Step 2 — Load the Same BeerQA Dev20 Questions

In [ ]:
with open(BEERQA_DEV_PATH, "r", encoding="utf-8") as f:
    beerqa_dev = json.load(f)

beerqa_records = beerqa_dev["data"]
beerqa_sample = beerqa_records[:DEV_SAMPLE_SIZE]

print("Total BeerQA records:", len(beerqa_records))
print("Router sample:", len(beerqa_sample))

## Step 3 — Define the Routing Rules

The router should choose the **simplest sufficient workflow**.

### EASY
Choose Standard RAG when a directly retrievable fact is likely enough and little intermediate reasoning is needed.

### MEDIUM
Choose Query Decomposition RAG when there are multiple separable information needs, comparisons, or compositional questions whose pieces can be retrieved independently and merged.

### HARD
Choose Multi-Agent RAG when the question is likely to require dependent multi-hop reasoning, iterative retrieval, replanning, or verification.

In [ ]:
ROUTING_POLICY = {
    "EASY": "STANDARD_RAG",
    "MEDIUM": "QUERY_DECOMPOSITION_RAG",
    "HARD": "MULTI_AGENT_RAG",
}

for complexity, route in ROUTING_POLICY.items():
    print(complexity, "->", route)

## Step 4 — Connect DeepSeek

In [ ]:
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass(
        "Enter your DeepSeek API key: "
    )

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

DEEPSEEK_MODEL = "deepseek-v4-flash"

print("DeepSeek router client ready.")

## Step 5 — Parse Router JSON

In [ ]:
def strip_json_fence(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def parse_router_output(raw_output):
    cleaned = strip_json_fence(raw_output)

    try:
        parsed = json.loads(cleaned)
    except Exception:
        parsed = {}

    complexity = str(parsed.get("complexity", "")).strip().upper()

    used_fallback = False

    if complexity not in ROUTING_POLICY:
        complexity = "MEDIUM"
        used_fallback = True

    route = ROUTING_POLICY[complexity]

    try:
        confidence = float(parsed.get("confidence", 0.0))
    except Exception:
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))

    signals = parsed.get("signals", {})
    if not isinstance(signals, dict):
        signals = {}

    return {
        "complexity": complexity,
        "route": route,
        "reason": str(parsed.get("reason", "")).strip(),
        "confidence": confidence,
        "signals": signals,
        "used_fallback": used_fallback,
        "raw_output": raw_output,
    }

## Step 6 — DeepSeek Complexity Router

In [ ]:
def complexity_router(question, model=DEEPSEEK_MODEL):

    system_prompt = """
You are the Complexity-Aware Router for an adaptive RAG system.

Your ONLY task is to classify the expected retrieval/reasoning
difficulty of the question and select the simplest sufficient route.

EASY -> STANDARD_RAG
Use when:
- a directly retrievable fact is likely sufficient;
- the question is mainly a direct entity/property lookup;
- minimal intermediate reasoning is required.

MEDIUM -> QUERY_DECOMPOSITION_RAG
Use when:
- two or more separable information needs exist;
- the question compares or combines independently retrievable facts;
- decomposition and one merged evidence stage are likely sufficient.

HARD -> MULTI_AGENT_RAG
Use when:
- later retrieval may depend on an earlier intermediate result;
- dependent multi-hop reasoning is required;
- iterative retrieval, replanning, or verification is likely useful;
- there is a long linked relational or temporal reasoning chain.

Rules:
1. Do NOT answer the question.
2. Use ONLY the question text.
3. Do NOT assume access to the BeerQA gold answer or context.
4. Prefer the simpler route when it is sufficient.
5. Return ONLY valid JSON.

Required format:
{
  "complexity": "EASY | MEDIUM | HARD",
  "route": "STANDARD_RAG | QUERY_DECOMPOSITION_RAG | MULTI_AGENT_RAG",
  "reason": "one short explanation",
  "confidence": 0.0,
  "signals": {
    "direct_fact": true,
    "multiple_information_needs": false,
    "comparison_or_composition": false,
    "dependent_hops": false,
    "iterative_verification_likely": false
  }
}
""".strip()

    user_prompt = f"""
Question:
{question}

Return the routing JSON only.
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=300,
        temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},
    )

    raw_output = response.choices[0].message.content.strip()

    return parse_router_output(raw_output)

## Step 7 — Test One Real BeerQA Question

In [ ]:
test_record = beerqa_sample[0]
test_question = test_record["question"]

print("Question:")
print(test_question)

router_test = complexity_router(test_question)

print("\nComplexity:", router_test["complexity"])
print("Route:", router_test["route"])
print("Confidence:", router_test["confidence"])
print("Reason:", router_test["reason"])
print("Signals:")
print(json.dumps(router_test["signals"], indent=2))
print("Fallback used:", router_test["used_fallback"])

## Step 8 — Batch Routing Function

In [ ]:
def route_records(records, limit=None):

    if limit is not None:
        records = records[:limit]

    decisions = []

    for i, record in enumerate(records, start=1):

        print(f"\nRouting {i} / {len(records)}")
        print("Question:", record["question"])

        try:
            decision = complexity_router(record["question"])

            row = {
                "id": record["id"],
                "src": record["src"],
                "question": record["question"],
                "complexity": decision["complexity"],
                "route": decision["route"],
                "reason": decision["reason"],
                "confidence": decision["confidence"],
                "signals": decision["signals"],
                "used_fallback": decision["used_fallback"],
                "error": None,
            }

            decisions.append(row)

            print(
                "Decision:",
                row["complexity"],
                "->",
                row["route"]
            )

        except Exception as e:

            print("ERROR:", e)

            decisions.append({
                "id": record.get("id"),
                "src": record.get("src"),
                "question": record.get("question"),
                "complexity": None,
                "route": None,
                "reason": None,
                "confidence": None,
                "signals": {},
                "used_fallback": None,
                "error": str(e),
            })

    return decisions

## Step 9 — Run Dev5 First

In [ ]:
if ROUTER_DEV5_PATH.exists():

    router_dev5 = []

    with open(ROUTER_DEV5_PATH, "r", encoding="utf-8") as f:
        for line in f:
            router_dev5.append(json.loads(line))

    print("Loaded saved Router Dev5:", len(router_dev5))

else:

    router_dev5 = route_records(
        beerqa_sample,
        limit=5
    )

    with open(ROUTER_DEV5_PATH, "w", encoding="utf-8") as f:
        for row in router_dev5:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Saved Router Dev5:", ROUTER_DEV5_PATH)

In [ ]:
successful_router5 = [
    row for row in router_dev5
    if row["error"] is None
]

print(
    "Route distribution:",
    dict(Counter(row["complexity"] for row in successful_router5))
)

print(
    "Fallback count:",
    sum(1 for row in successful_router5 if row["used_fallback"])
)

## Step 10 — Prepare Dev20 Routing

In [ ]:
if ROUTER_DEV20_PATH.exists():

    router_dev20 = []

    with open(ROUTER_DEV20_PATH, "r", encoding="utf-8") as f:
        for line in f:
            router_dev20.append(json.loads(line))

    print("Loaded saved Router Dev20:", len(router_dev20))

else:

    router_dev20 = []

    print("No Dev20 router decisions yet.")
    print("After checking Dev5, run the next cell intentionally.")

In [ ]:
# RUN THIS CELL ONLY WHEN READY FOR ALL 20 ROUTER CALLS

# router_dev20 = route_records(
#     beerqa_sample,
#     limit=DEV_SAMPLE_SIZE
# )
#
# with open(ROUTER_DEV20_PATH, "w", encoding="utf-8") as f:
#     for row in router_dev20:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")
#
# print("Saved Router Dev20:", ROUTER_DEV20_PATH)

## Step 11 — Inspect Router Distribution

In [ ]:
if router_dev20:

    successful_router20 = [
        row for row in router_dev20
        if row["error"] is None
    ]

    distribution = Counter(
        row["complexity"]
        for row in successful_router20
    )

    average_confidence = sum(
        row["confidence"]
        for row in successful_router20
    ) / len(successful_router20)

    fallback_count = sum(
        1
        for row in successful_router20
        if row["used_fallback"]
    )

    print("Questions routed:", len(successful_router20))
    print("Distribution:", dict(distribution))
    print("Average confidence:", round(average_confidence, 4))
    print("Fallback count:", fallback_count)

# Adaptive Dev20 Evaluation

The three RAG pipelines are already evaluated independently.  
Now we use the router decision to choose the corresponding saved pipeline output for each question.

## Step 12 — Load Saved Pipeline Results

In [ ]:
def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows


required_files = [
    STANDARD_RAG_EVAL_PATH,
    QD_RAG_EVAL_PATH,
    MA_RAG_EVAL_PATH,
]

missing = [
    path for path in required_files
    if not path.exists()
]

if missing:
    print("Missing evaluated result files:")
    for path in missing:
        print("-", path)

    raise FileNotFoundError(
        "All three evaluated Dev20 result files are required."
    )


standard_rows = load_jsonl(STANDARD_RAG_EVAL_PATH)
qd_rows = load_jsonl(QD_RAG_EVAL_PATH)
ma_rows = load_jsonl(MA_RAG_EVAL_PATH)

print("Standard RAG rows:", len(standard_rows))
print("QD RAG rows:", len(qd_rows))
print("Multi-Agent rows:", len(ma_rows))

## Step 13 — Validate Same Questions Across All Three Pipelines

In [ ]:
standard_ids = {row["id"] for row in standard_rows}
qd_ids = {row["id"] for row in qd_rows}
ma_ids = {row["id"] for row in ma_rows}
beerqa_ids = {record["id"] for record in beerqa_sample}

assert standard_ids == qd_ids
assert standard_ids == ma_ids
assert standard_ids == beerqa_ids

print("PASS: all three pipelines use the same Dev20 questions.")

## Step 14 — Create Pipeline Lookup Tables

In [ ]:
PIPELINE_RESULTS = {
    "STANDARD_RAG": {
        row["id"]: row
        for row in standard_rows
    },

    "QUERY_DECOMPOSITION_RAG": {
        row["id"]: row
        for row in qd_rows
    },

    "MULTI_AGENT_RAG": {
        row["id"]: row
        for row in ma_rows
    },
}

print("Pipeline lookup tables ready.")

## Step 15 — Build Adaptive Results

In [ ]:
def build_adaptive_results(router_decisions):

    adaptive_rows = []

    for decision in router_decisions:

        if decision.get("error") is not None:
            adaptive_rows.append({
                "id": decision.get("id"),
                "question": decision.get("question"),
                "error": "Router failed: " + str(decision["error"]),
            })
            continue

        qid = decision["id"]
        selected_route = decision["route"]

        selected_result = (
            PIPELINE_RESULTS[
                selected_route
            ][qid]
        )

        adaptive_rows.append({
            "id": qid,
            "src": selected_result.get("src"),
            "question": decision["question"],

            "router_complexity": decision["complexity"],
            "selected_route": selected_route,
            "router_reason": decision["reason"],
            "router_confidence": decision["confidence"],
            "router_signals": decision["signals"],
            "router_fallback": decision["used_fallback"],

            "gold_answers": selected_result.get("gold_answers"),
            "prediction": selected_result.get("prediction"),

            "em": selected_result.get("em"),
            "f1": selected_result.get("f1"),

            "title_recall@10":
                selected_result.get("title_recall@10"),

            "title_precision@10":
                selected_result.get("title_precision@10"),

            "retrieved_passages":
                selected_result.get("retrieved_passages", []),

            "pipeline_error":
                selected_result.get("error"),

            "error": None,
        })

    return adaptive_rows


if router_dev20:
    adaptive_dev20 = build_adaptive_results(router_dev20)
    print("Adaptive rows:", len(adaptive_dev20))
else:
    adaptive_dev20 = []
    print("Run/load Router Dev20 first.")

## Step 16 — Adaptive EM / F1 / Retrieval Summary

In [ ]:
def summarize_rows(rows):

    successful = [
        row for row in rows
        if (
            row.get("error") is None
            and row.get("pipeline_error") is None
        )
    ]

    if not successful:
        return None

    summary = {
        "n": len(successful),

        "em": sum(
            row["em"]
            for row in successful
        ) / len(successful),

        "f1": sum(
            row["f1"]
            for row in successful
        ) / len(successful),
    }

    recall_values = [
        row["title_recall@10"]
        for row in successful
        if row.get("title_recall@10") is not None
    ]

    precision_values = [
        row["title_precision@10"]
        for row in successful
        if row.get("title_precision@10") is not None
    ]

    if recall_values:
        summary["title_recall@10"] = (
            sum(recall_values) / len(recall_values)
        )

    if precision_values:
        summary["title_precision@10"] = (
            sum(precision_values) / len(precision_values)
        )

    return summary


if adaptive_dev20:

    adaptive_summary = summarize_rows(
        adaptive_dev20
    )

    print("Questions:", adaptive_summary["n"])
    print("Average EM:", round(adaptive_summary["em"], 4))
    print("Average F1:", round(adaptive_summary["f1"], 4))
    print(
        "Average Title Recall@10:",
        round(adaptive_summary.get("title_recall@10", float("nan")), 4)
    )
    print(
        "Average Title Precision@10:",
        round(adaptive_summary.get("title_precision@10", float("nan")), 4)
    )

## Step 17 — Save Adaptive Dev20 Results

In [ ]:
if adaptive_dev20:

    with open(
        ADAPTIVE_DEV20_EVAL_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        for row in adaptive_dev20:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

    print(
        "Saved adaptive evaluated results:",
        ADAPTIVE_DEV20_EVAL_PATH
    )

## Step 18 — Compare All Four Systems

In [ ]:
def summarize_pipeline(rows):

    successful = [
        row for row in rows
        if row.get("error") is None
    ]

    out = {
        "em": sum(row["em"] for row in successful) / len(successful),
        "f1": sum(row["f1"] for row in successful) / len(successful),
    }

    recalls = [
        row["title_recall@10"]
        for row in successful
        if row.get("title_recall@10") is not None
    ]

    precisions = [
        row["title_precision@10"]
        for row in successful
        if row.get("title_precision@10") is not None
    ]

    if recalls:
        out["title_recall@10"] = sum(recalls) / len(recalls)

    if precisions:
        out["title_precision@10"] = sum(precisions) / len(precisions)

    return out


if adaptive_dev20:

    comparison = {
        "Standard RAG":
            summarize_pipeline(standard_rows),

        "Query Decomposition RAG":
            summarize_pipeline(qd_rows),

        "Multi-Agent RAG":
            summarize_pipeline(ma_rows),

        "Adaptive Router Framework":
            adaptive_summary,
    }

    print("=" * 112)
    print(
        f"{'System':<30}"
        f"{'EM':>12}"
        f"{'F1':>12}"
        f"{'Recall@10':>20}"
        f"{'Precision@10':>22}"
    )
    print("=" * 112)

    for name, summary in comparison.items():

        print(
            f"{name:<30}"
            f"{summary.get('em', float('nan')):>12.4f}"
            f"{summary.get('f1', float('nan')):>12.4f}"
            f"{summary.get('title_recall@10', float('nan')):>20.4f}"
            f"{summary.get('title_precision@10', float('nan')):>22.4f}"
        )

## Step 19 — Inspect Every Routing Decision

In [ ]:
if adaptive_dev20:

    for row in adaptive_dev20:

        print("=" * 100)

        print("Question:", row["question"])
        print("Complexity:", row["router_complexity"])
        print("Selected route:", row["selected_route"])
        print("Confidence:", row["router_confidence"])
        print("Reason:", row["router_reason"])
        print("Gold:", row["gold_answers"])
        print("Prediction:", row["prediction"])
        print("EM:", row["em"])
        print("F1:", round(row["f1"], 4))

## Step 20 — Diagnostic: Compare Router Choice with Best Observed Pipeline

BeerQA does not provide gold EASY/MEDIUM/HARD labels here.

For analysis only, compare the router-selected pipeline against the pipeline that achieved the best observed answer score for each question.

This information is **never given to the router**.

In [ ]:
standard_by_id = {
    row["id"]: row
    for row in standard_rows
}

qd_by_id = {
    row["id"]: row
    for row in qd_rows
}

ma_by_id = {
    row["id"]: row
    for row in ma_rows
}


def best_observed_route(qid):

    options = {
        "STANDARD_RAG":
            standard_by_id[qid],

        "QUERY_DECOMPOSITION_RAG":
            qd_by_id[qid],

        "MULTI_AGENT_RAG":
            ma_by_id[qid],
    }

    return max(
        options.items(),
        key=lambda item: (
            item[1].get("em", 0),
            item[1].get("f1", 0.0)
        )
    )


if adaptive_dev20:

    matched = 0

    for row in adaptive_dev20:

        best_route, best_row = (
            best_observed_route(
                row["id"]
            )
        )

        if row["selected_route"] == best_route:
            matched += 1

        print("=" * 100)

        print("Question:", row["question"])
        print("Router selected:", row["selected_route"])
        print("Best observed route:", best_route)

        print(
            "Selected EM/F1:",
            row["em"],
            "/",
            round(row["f1"], 4)
        )

        print(
            "Best observed EM/F1:",
            best_row["em"],
            "/",
            round(best_row["f1"], 4)
        )

    print(
        "\nRouter selected the best-observed route:",
        matched,
        "/",
        len(adaptive_dev20)
    )

## Step 21 — Optional Live Dispatcher

For the current Dev20 evaluation, using saved pipeline results is recommended.

Later, after the three pipeline functions are moved into reusable `.py` modules, the router can directly call the selected function.

Expected functions:
- `beerqa_standard_rag(question)`
- `query_decomposition_rag(question)`
- `multi_agent_rag(question)`

In [ ]:
def live_adaptive_rag(question):

    decision = complexity_router(
        question
    )

    route = decision["route"]

    if route == "STANDARD_RAG":

        if "beerqa_standard_rag" not in globals():
            raise RuntimeError(
                "beerqa_standard_rag() is not loaded."
            )

        result = beerqa_standard_rag(
            question
        )

    elif route == "QUERY_DECOMPOSITION_RAG":

        if "query_decomposition_rag" not in globals():
            raise RuntimeError(
                "query_decomposition_rag() is not loaded."
            )

        result = query_decomposition_rag(
            question
        )

    elif route == "MULTI_AGENT_RAG":

        if "multi_agent_rag" not in globals():
            raise RuntimeError(
                "multi_agent_rag() is not loaded."
            )

        result = multi_agent_rag(
            question
        )

    else:
        raise ValueError(
            f"Unknown route: {route}"
        )

    return {
        "router": decision,
        "selected_route": route,
        "pipeline_result": result,
    }

# Completion Checklist

The complexity-aware router development stage is complete when:

- [ ] One-question routing test works.
- [ ] EASY maps to Standard RAG.
- [ ] MEDIUM maps to Query Decomposition RAG.
- [ ] HARD maps to Multi-Agent RAG.
- [ ] Router sees only the original question.
- [ ] Dev5 decisions look sensible.
- [ ] Dev20 router decisions are saved.
- [ ] All three evaluated pipeline result files are present.
- [ ] Adaptive Dev20 results are generated.
- [ ] Adaptive EM and F1 are calculated.
- [ ] Adaptive Title Recall@10 and Title Precision@10 are calculated.
- [ ] Four-system comparison is produced.
- [ ] Router distribution, confidence, and fallback rate are inspected.
- [ ] Router choices are analysed against the best observed pipeline for discussion.

After this development notebook is complete, the next major FYP stage is the final experimental setup, including a larger/common corpus, final evaluation metrics, and ablation analysis.